# 02 — Silver Layer: Tratamento, Normalização e Qualidade

Trata e normaliza os dados da camada Bronze. Cada entidade recebe tratamento específico com base nos problemas identificados na exploração.

| Fonte | Problema principal | Tratamento |
|---|---|---|
| pedidos_cabecalho | datas em 3 formatos, status inconsistente, 64 nulos | parse multi-formato, normalização |
| pedidos_itens | item_status nulo (26%), unit_price string, 77 mismatches total_item | imputação, recálculo |
| clientes | estado com 18 variações, porte/status com case inconsistente | mapa de normalização |
| canais | CH05 duplicado, CH06 sem nome, ch07 lowercase, ativo variado | deduplicação, flags |
| vendedores | V004/V008 duplicados, 'sul' → 'S', CH99 inválido | deduplicação com critério |
| produtos | 1 product_id duplicado, 21 status nulos | manter mais recente |
| logística | delivery_status com variações e 60 nulos | normalização |
| ocorrências | event_type/severity/status com variações e nulos | normalização |

## Setup

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType

spark.sql("USE CATALOG sandbox")
spark.sql("CREATE DATABASE IF NOT EXISTS silver COMMENT 'Camada Silver — dados normalizados e com qualidade tratada'")
print("sandbox.silver pronto")

sandbox.silver pronto


## Helper: Parsing de Datas Multi-Formato

Três formatos identificados nas fontes: `yyyy-MM-dd`, `yyyy/MM/dd`, `dd/MM/yyyy`.

In [0]:
def parse_date_multiformat(col_name):
    return F.coalesce(
        F.expr(f"try_to_date(`{col_name}`, 'yyyy-MM-dd')"),
        F.expr(f"try_to_date(`{col_name}`, 'yyyy/MM/dd')"),
        F.expr(f"try_to_date(`{col_name}`, 'dd/MM/yyyy')"),
    )

print("Helper atualizado")

Helper atualizado


## 1. silver.dim_regioes

Normaliza regional_code para uppercase, remove entradas inativas (active_flag=0) e a entrada inválida `XX`. Deduplicação mantém o nome mais curto (mais canônico) por código.

In [0]:
df_reg = spark.table("bronze.legado_regioes")

df_reg_silver = (
    df_reg
    .withColumn("regional_code", F.trim(F.upper(F.col("regional_code"))))
    .withColumn("regional_name",  F.trim(F.col("regional_name")))
    .withColumn("state",          F.trim(F.upper(F.col("state"))))
    .withColumn("active_flag",    F.col("active_flag").cast("int"))
    .filter(
        F.col("regional_code").isNotNull() &
        (F.col("regional_code") != "XX") &
        (F.col("active_flag") == 1)
    )
    .withColumn("_rn", F.row_number().over(
        Window.partitionBy("regional_code").orderBy(F.length("regional_name"))
    ))
    .filter(F.col("_rn") == 1)
    .drop("_rn", "_ingested_at", "_source_file")
    .withColumn("_processed_at", F.current_timestamp())
)

df_reg_silver.write.format("delta").mode("overwrite").saveAsTable("silver.dim_regioes")
print(f"silver.dim_regioes: {df_reg_silver.count()} registros")
df_reg_silver.show(truncate=False)

silver.dim_regioes: 6 registros
+-------------+-------------+------------+--------------+-----------+--------------------------+
|regional_code|regional_name|state       |manager_name  |active_flag|_processed_at             |
+-------------+-------------+------------+--------------+-----------+--------------------------+
|CO           |Centro-Oeste |GO          |Paulo Teixeira|1          |2026-06-09 20:06:56.405367|
|N            |Norte        |AM          |Mariana Lopes |1          |2026-06-09 20:06:56.405367|
|NE           |Nordeste     |BA          |Carlos Lima   |1          |2026-06-09 20:06:56.405367|
|S            |Sul          |SC          |Rafael Souza  |1          |2026-06-09 20:06:56.405367|
|SE           |Sudeste      |SP          |Ana Costa     |1          |2026-06-09 20:06:56.405367|
|SUL          |Região Sul   |STA CATARINA|Rafael Souza  |1          |2026-06-09 20:06:56.405367|
+-------------+-------------+------------+--------------+-----------+--------------------------

## 2. silver.dim_canais

Normaliza id_canal para uppercase, converte `ativo` para boolean, trata CH05 duplicado (mantém primeiro), CH06 sem nome e ch07 em lowercase.

In [0]:
df_can = spark.table("bronze.comercial_canais")

df_can_silver = (
    df_can
    .withColumn("id_canal",   F.trim(F.upper(F.col("id_canal"))))
    .withColumn("nome_canal", F.when(F.col("nome_canal").isNull(), "NOME_AUSENTE")
                               .otherwise(F.trim(F.col("nome_canal"))))
    .withColumn("tipo_canal", F.trim(F.upper(F.col("tipo_canal"))))
    .withColumn("ativo_flag",
        F.when(F.lower(F.trim(F.col("ativo"))).isin("sim","yes","1","true"), True)
         .when(F.lower(F.trim(F.col("ativo"))).isin("nao","não","no","0","false"), False)
         .otherwise(None)
    )
    # CH05 duplicado: manter registro sem observação de "duplicado"
    .withColumn("_rn", F.row_number().over(
        Window.partitionBy("id_canal").orderBy(
            F.when(F.col("observacao").isNull(), 0).otherwise(1)
        )
    ))
    .filter(F.col("_rn") == 1)
    .drop("_rn", "ativo", "_ingested_at", "_source_file")
    .withColumn("_processed_at", F.current_timestamp())
)

df_can_silver.write.format("delta").mode("overwrite").saveAsTable("silver.dim_canais")
print(f"silver.dim_canais: {df_can_silver.count()} registros")
df_can_silver.show(truncate=False)

silver.dim_canais: 7 registros
+--------+------------+----------+---------------+----------+-------------------------+
|id_canal|nome_canal  |tipo_canal|observacao     |ativo_flag|_processed_at            |
+--------+------------+----------+---------------+----------+-------------------------+
|CH01    |Inside Sales|DIRETO    |NULL           |true      |2026-06-09 20:07:00.90672|
|CH02    |Parceiros   |INDIRETO  |NULL           |true      |2026-06-09 20:07:00.90672|
|CH03    |Marketplace |DIGITAL   |NULL           |true      |2026-06-09 20:07:00.90672|
|CH04    |Field Sales |DIRETO    |canal legado   |false     |2026-06-09 20:07:00.90672|
|CH05    |E-commerce  |DIGITAL   |NULL           |true      |2026-06-09 20:07:00.90672|
|CH06    |NOME_AUSENTE|INDIRETO  |nome ausente   |true      |2026-06-09 20:07:00.90672|
|CH07    |Revendas    |INDIRETO  |id em lowercase|NULL      |2026-06-09 20:07:00.90672|
+--------+------------+----------+---------------+----------+-------------------------+



## 3. silver.dim_produtos

Achata estrutura JSON aninhada, normaliza status, deduplica pelo product_id mantendo o registro com updated_at mais recente.

In [0]:
df_prod = spark.table("bronze.cadastro_produtos")

df_prod_silver = (
    df_prod
    .select(
        F.col("product.product_id").alias("product_id"),
        F.col("product.name").alias("product_name"),
        F.col("product.category").alias("category"),
        F.col("product.subcategory").alias("subcategory"),
        F.col("product.status").alias("status_raw"),
        F.col("pricing.list_price").alias("list_price"),
        F.col("pricing.currency").alias("currency"),
        F.col("attributes.family").alias("family"),
        F.col("attributes.tags").alias("tags"),
        F.to_timestamp(F.col("updated_at")).alias("updated_at"),
    )
    .withColumn("status_produto",
        F.when(F.lower(F.trim(F.col("status_raw"))).isin("ativo","active"), "Ativo")
         .when(F.lower(F.trim(F.col("status_raw"))) == "inativo", "Inativo")
         .when(F.lower(F.trim(F.col("status_raw"))) == "descontinuado", "Descontinuado")
         .when(F.col("status_raw").isNull(), "Desconhecido")
         .otherwise(F.col("status_raw"))
    )
    # Deduplicar: manter registro mais recente por product_id
    .withColumn("_rn", F.row_number().over(
        Window.partitionBy("product_id").orderBy(F.col("updated_at").desc())
    ))
    .filter(F.col("_rn") == 1)
    .drop("_rn", "status_raw")
    .withColumn("_processed_at", F.current_timestamp())
)

df_prod_silver.write.format("delta").mode("overwrite").saveAsTable("silver.dim_produtos")
print(f"silver.dim_produtos: {df_prod_silver.count()} registros")
df_prod_silver.select("product_id","product_name","category","status_produto").show(5)

silver.dim_produtos: 71 registros
+----------+------------+----------+--------------+
|product_id|product_name|  category|status_produto|
+----------+------------+----------+--------------+
|     P0001|   Produto 1|  Software|         Ativo|
|     P0002|   Produto 2|  Hardware|         Ativo|
|     P0003|   Produto 3|  Software|  Desconhecido|
|     P0004|   Produto 4|Assinatura|         Ativo|
|     P0005|   Produto 5|Assinatura|       Inativo|
+----------+------------+----------+--------------+
only showing top 5 rows


## 4. silver.dim_clientes

Normaliza estado (18 variações → sigla UF), porte e status. Faz parse de data_cadastro com multi-formato. Registros com data inválida recebem flag.

In [0]:
# Mapa explícito de normalização de estado
estado_when = F.col("estado")
for k, v in {
    "santa catarina": "SC", "Santa Catarina": "SC", "Sta Catarina": "SC", "S. Catarina": "SC",
    "Paraná": "PR", "parana": "PR",
    "Rio de Janeiro": "RJ", "rio de janeiro": "RJ",
    "Minas Gerais": "MG", "minas gerais": "MG",
    "Sao Paulo": "SP", "sao paulo": "SP", "São Paulo": "SP",
}.items():
    estado_when = F.when(F.col("estado") == k, v).otherwise(estado_when)

df_cli = spark.table("bronze.crm_clientes")

df_cli_silver = (
    df_cli
    .withColumn("estado_norm", F.upper(F.trim(estado_when)))
    .withColumn("porte_norm",
        F.when(F.lower(F.trim(F.col("porte"))).isin("grande","large"), "Grande")
         .when(F.lower(F.trim(F.col("porte"))).isin("média","media","medium"), "Média")
         .when(F.lower(F.trim(F.col("porte"))).isin("pequena","small"), "Pequena")
         .otherwise(None)
    )
    .withColumn("status_cliente_norm",
        F.when(F.lower(F.trim(F.col("status_cliente"))).isin("ativo","active"), "Ativo")
         .when(F.lower(F.trim(F.col("status_cliente"))).isin("inativo","inactive"), "Inativo")
         .otherwise("Desconhecido")
    )
    .withColumn("data_cadastro_dt", parse_date_multiformat("data_cadastro"))
    .withColumn("data_cadastro_flag_invalida", F.col("data_cadastro_dt").isNull())
    .select(
        F.col("customer_id"),
        F.col("nome_cliente"),
        F.col("segmento"),
        F.col("porte_norm").alias("porte"),
        F.col("cidade"),
        F.col("estado_norm").alias("estado"),
        F.col("data_cadastro_dt").alias("data_cadastro"),
        F.col("data_cadastro_flag_invalida"),
        F.col("email"),
        F.col("status_cliente_norm").alias("status_cliente"),
        F.expr("try_cast(replace(updated_at, ',', '.') AS TIMESTAMP)").alias("updated_at"),
        F.current_timestamp().alias("_processed_at"),
    )
)

df_cli_silver.write.format("delta").mode("overwrite").saveAsTable("silver.dim_clientes")
print(f"silver.dim_clientes: {df_cli_silver.count()} registros")
print(f"  datas inválidas: {df_cli_silver.filter('data_cadastro_flag_invalida').count()}")
df_cli_silver.groupBy("estado").count().orderBy("estado").show(20)

silver.dim_clientes: 183 registros
  datas inválidas: 0
+------+-----+
|estado|count|
+------+-----+
|    MG|   39|
|    PR|   31|
|    RJ|   50|
|    SC|   29|
|    SP|   34|
+------+-----+



## 5. silver.dim_vendedores

Normaliza regional_code ('sul' → 'S'), canal_id para uppercase, status. Deduplica V004 (mantém canal válido) e V008 (descarta duplicata). Sinaliza CH99 como inválido.

In [0]:
df_vend = spark.table("bronze.vendedores")

df_vend_silver = (
    df_vend
    .withColumn("regional_code_norm",
        F.when(F.lower(F.trim(F.col("regional_code"))) == "sul", "S")
         .otherwise(F.upper(F.trim(F.col("regional_code"))))
    )
    .withColumn("canal_id_norm", F.upper(F.trim(F.col("canal_id"))))
    .withColumn("status_norm",
        F.when(F.lower(F.trim(F.col("status"))).isin("ativo","active"), "Ativo")
         .when(F.lower(F.trim(F.col("status"))).isin("inativo","inactive"), "Inativo")
         .otherwise("Desconhecido")
    )
    .withColumn("hire_date_dt", parse_date_multiformat("hire_date"))
    .withColumn("flag_canal_invalido",
        F.when(F.upper(F.trim(F.col("canal_id"))) == "CH99", True).otherwise(False)
    )
    # Deduplicar: por seller_id, preferir canal válido; em empate, o mais antigo
    .withColumn("_rn", F.row_number().over(
        Window.partitionBy("seller_id").orderBy(
            F.col("flag_canal_invalido").asc(),
            F.col("hire_date_dt").asc()
        )
    ))
    .filter(F.col("_rn") == 1)
    .select(
        F.col("seller_id"),
        F.col("seller_name"),
        F.col("canal_id_norm").alias("canal_id"),
        F.col("regional_code_norm").alias("regional_code"),
        F.col("hire_date_dt").alias("hire_date"),
        F.col("status_norm").alias("status"),
        F.col("flag_canal_invalido"),
        F.current_timestamp().alias("_processed_at"),
    )
)

df_vend_silver.write.format("delta").mode("overwrite").saveAsTable("silver.dim_vendedores")
print(f"silver.dim_vendedores: {df_vend_silver.count()} registros")
df_vend_silver.show(truncate=False)

silver.dim_vendedores: 40 registros
+---------+-----------+--------+-------------+----------+------------+-------------------+--------------------------+
|seller_id|seller_name|canal_id|regional_code|hire_date |status      |flag_canal_invalido|_processed_at             |
+---------+-----------+--------+-------------+----------+------------+-------------------+--------------------------+
|V001     |Vendedor 1 |CH01    |S            |2024-06-27|Desconhecido|false              |2026-06-09 20:07:14.034444|
|V002     |Vendedor 2 |CH04    |S            |2023-12-16|Ativo       |false              |2026-06-09 20:07:14.034444|
|V003     |Vendedor 3 |CH03    |S            |2023-11-29|Ativo       |false              |2026-06-09 20:07:14.034444|
|V004     |Vendedor 4 |CH02    |NE           |2025-02-11|Ativo       |false              |2026-06-09 20:07:14.034444|
|V005     |Vendedor 5 |CH03    |S            |2024-11-08|Desconhecido|false              |2026-06-09 20:07:14.034444|
|V006     |Vendedor 

## 6. silver.fct_pedidos_cabecalho

Normaliza datas, status_order, extrai campos do JSON payment_details. Imputa gross_amount nulo pela soma dos itens (via join). Cria flags de status.

In [0]:
df_cab   = spark.table("bronze.erp_pedidos_cabecalho")
df_itens = spark.table("bronze.erp_pedidos_itens")

# Agregar itens para imputação de gross_amount
df_itens_agg = (
    df_itens
    .withColumn("unit_price_num", F.expr("try_cast(replace(unit_price, ',', '.') AS DOUBLE)"))
    .withColumn("quantity_num",   F.expr("try_cast(replace(quantity,   ',', '.') AS DOUBLE)"))
    .withColumn("total_calc",     F.round(F.col("quantity_num") * F.col("unit_price_num"), 2))
    .groupBy("order_id")
    .agg(F.sum("total_calc").alias("soma_itens"),
         F.count("*").alias("qtd_itens"),
         F.countDistinct("product_code").alias("qtd_produtos_distintos"))
)

df_cab_silver = (
    df_cab
    .withColumn("order_date_dt",    parse_date_multiformat("order_date"))
    .withColumn("promised_date_dt", parse_date_multiformat("promised_date"))
    .withColumn("status_order_norm",
        F.upper(F.regexp_replace(F.trim(F.col("status_order")), "_", " "))
    )
    .withColumn("flag_status_nulo", F.col("status_order").isNull())
    .withColumn("gross_amount_num",    F.expr("try_cast(replace(gross_amount,    ',', '.') AS DOUBLE)"))
    .withColumn("discount_amount_num", F.expr("try_cast(replace(discount_amount, ',', '.') AS DOUBLE)"))
    .withColumn("net_amount_num",      F.expr("try_cast(replace(net_amount,      ',', '.') AS DOUBLE)"))
    .withColumn("payment_source",   F.get_json_object(F.col("payment_details"), "$.source"))
    .withColumn("payment_priority", F.get_json_object(F.col("payment_details"), "$.priority"))
    .withColumn("last_update_dt", F.expr("try_to_timestamp(last_update)"))
    .join(df_itens_agg, on="order_id", how="left")
    .withColumn("gross_amount_final",
        F.coalesce(F.col("gross_amount_num"), F.col("soma_itens"))
    )
    .select(
        "order_id", "customer_code", "seller_id",
        F.col("order_date_dt").alias("order_date"),
        F.col("promised_date_dt").alias("promised_date"),
        F.col("status_order_norm").alias("status_order"),
        F.col("flag_status_nulo"),
        F.col("gross_amount_final").alias("gross_amount"),
        F.col("discount_amount_num").alias("discount_amount"),
        F.col("net_amount_num").alias("net_amount"),
        "payment_source", "payment_priority",
        F.col("last_update_dt").alias("last_update"),
        "soma_itens", "qtd_itens", "qtd_produtos_distintos",
        F.current_timestamp().alias("_processed_at"),
    )
)

df_cab_silver.write.format("delta").mode("overwrite").saveAsTable("silver.fct_pedidos_cabecalho")
print(f"silver.fct_pedidos_cabecalho: {df_cab_silver.count()} registros")
df_cab_silver.groupBy("status_order").count().orderBy("count", ascending=False).show()

silver.fct_pedidos_cabecalho: 403 registros
+------------+-----+
|status_order|count|
+------------+-----+
|EM SEPARACAO|  124|
|    FATURADO|  106|
|        NULL|   64|
|    ENTREGUE|   56|
|   CANCELADO|   53|
+------------+-----+



## 7. silver.fct_pedidos_itens

Normaliza item_status (nulo → 'Ativo'), converte unit_price para double, recalcula total_item, sinaliza divergências.

In [0]:
df_itens = spark.table("bronze.erp_pedidos_itens")

df_itens_silver = (
    df_itens
    .withColumn("unit_price_num", F.expr("try_cast(replace(unit_price, ',', '.') AS DOUBLE)"))
    .withColumn("quantity_num",   F.expr("try_cast(replace(quantity,   ',', '.') AS DOUBLE)"))
    .withColumn("total_item_orig",F.expr("try_cast(replace(total_item, ',', '.') AS DOUBLE)"))
    .withColumn("item_status_norm",
        F.when(F.lower(F.trim(F.col("item_status"))) == "ativo", "Ativo")
         .when(F.lower(F.trim(F.col("item_status"))) == "cancelado", "Cancelado")
         .when(F.col("item_status").isNull(), "Ativo")
         .otherwise(F.col("item_status"))
    )
    .withColumn("total_item_calc", F.round(F.col("quantity_num") * F.col("unit_price_num"), 2))
    .withColumn("flag_total_divergente",
        F.when(
            F.col("total_item_orig").isNull() | F.col("total_item_calc").isNull(), False
        ).otherwise(
            F.abs(F.col("total_item_orig") - F.col("total_item_calc")) > 0.01
        )
    )
    .select(
        "order_id", "item_seq", "product_code",
        F.col("quantity_num").alias("quantity"),
        F.col("unit_price_num").alias("unit_price"),
        F.col("total_item_calc").alias("total_item"),
        F.col("item_status_norm").alias("item_status"),
        F.col("flag_total_divergente"),
        F.current_timestamp().alias("_processed_at"),
    )
)

df_itens_silver.write.format("delta").mode("overwrite").saveAsTable("silver.fct_pedidos_itens")
print(f"silver.fct_pedidos_itens: {df_itens_silver.count()} registros")
print(f"  itens com total divergente: {df_itens_silver.filter('flag_total_divergente').count()}")
df_itens_silver.groupBy("item_status").count().show()

silver.fct_pedidos_itens: 995 registros
  itens com total divergente: 47
+-----------+-----+
|item_status|count|
+-----------+-----+
|      Ativo|  752|
|  Cancelado|  243|
+-----------+-----+



## 8. silver.fct_entregas

Achata estrutura aninhada (carrier, timestamps, destination), normaliza delivery_status, calcula dias_transito, sinaliza carrier ausente.

In [0]:
df_ent = spark.table("bronze.logistica_entregas")

df_ent_silver = (
    df_ent
    .select(
        F.col("delivery_id"),
        F.col("order_ref").alias("order_id"),
        F.col("carrier.name").alias("carrier_name"),
        F.col("carrier.mode").alias("carrier_mode"),
        F.col("delivery_status").alias("status_raw"),
        F.expr("try_to_timestamp(timestamps.shipped_at,   'dd/MM/yyyy HH:mm')").alias("shipped_at"),
        F.expr("try_to_timestamp(timestamps.delivered_at, 'dd/MM/yyyy HH:mm')").alias("delivered_at"),
        F.col("destination.state").alias("dest_state"),
        F.col("destination.city").alias("dest_city"),
        F.expr("try_cast(replace(cost, ',', '.') AS DOUBLE)").alias("shipping_cost"),
    )
    .withColumn("delivery_status_norm",
        F.when(F.lower(F.trim(F.col("status_raw"))).isin("delivered","entregue"), "Entregue")
         .when(F.lower(F.trim(F.col("status_raw"))) == "in_transit",  "Em Trânsito")
         .when(F.lower(F.trim(F.col("status_raw"))) == "cancelled",   "Cancelado")
         .when(F.lower(F.trim(F.col("status_raw"))) == "atrasado",    "Atrasado")
         .when(F.col("status_raw").isNull(), "Desconhecido")
         .otherwise(F.col("status_raw"))
    )
    .withColumn("flag_carrier_ausente", F.col("carrier_name").isNull())
    .withColumn("dias_transito", F.datediff(F.col("delivered_at"), F.col("shipped_at")))
    .drop("status_raw")
    .withColumn("_processed_at", F.current_timestamp())
)

df_ent_silver.write.format("delta").mode("overwrite").saveAsTable("silver.fct_entregas")
print(f"silver.fct_entregas: {df_ent_silver.count()} registros")
df_ent_silver.groupBy("delivery_status_norm").count().orderBy("count", ascending=False).show()

silver.fct_entregas: 322 registros
+--------------------+-----+
|delivery_status_norm|count|
+--------------------+-----+
|            Entregue|  105|
|        Desconhecido|   60|
|         Em Trânsito|   60|
|           Cancelado|   53|
|            Atrasado|   44|
+--------------------+-----+



## 9. silver.fct_ocorrencias

Normaliza event_type, severity e status. Converte created_at para timestamp.

In [0]:
df_oc = spark.table("bronze.atendimento_ocorrencias")

df_oc_silver = (
    df_oc
    .withColumn("event_type_norm",
        F.when(F.lower(F.trim(F.col("event_type"))) == "delay",          "Atraso")
         .when(F.lower(F.trim(F.col("event_type"))) == "refund",         "Reembolso")
         .when(F.lower(F.trim(F.col("event_type"))) == "troca",          "Troca")
         .when(F.lower(F.trim(F.col("event_type"))) == "cancel_request", "Cancelamento")
         .when(F.lower(F.trim(F.col("event_type"))) == "complaint",      "Reclamação")
         .when(F.col("event_type").isNull(), "Desconhecido")
         .otherwise(F.col("event_type"))
    )
    .withColumn("severity_norm",
        F.when(F.lower(F.trim(F.col("severity"))) == "high",   "Alta")
         .when(F.lower(F.trim(F.col("severity"))) == "medium", "Média")
         .when(F.lower(F.trim(F.col("severity"))) == "low",    "Baixa")
         .when(F.col("severity").isNull(), "Desconhecida")
         .otherwise(F.col("severity"))
    )
    .withColumn("status_norm",
        F.when(F.lower(F.trim(F.col("status"))) == "open",   "Aberto")
         .when(F.lower(F.trim(F.col("status"))) == "closed", "Fechado")
         .when(F.col("status").isNull(), "Desconhecido")
         .otherwise(F.col("status"))
    )
    .withColumn("created_at_dt",
        F.coalesce(
            F.expr("try_to_timestamp(created_at, 'yyyy-MM-dd HH:mm:ss')"),
            F.expr("try_to_timestamp(created_at, 'yyyy/MM/dd HH:mm:ss')"),
            F.expr("try_to_timestamp(created_at, 'yyyy-MM-dd')"),
            F.expr("try_to_timestamp(created_at, 'yyyy/MM/dd')"),
        )
    )
    .select(
        F.col("ticket_id"),
        F.col("order_id"),
        F.col("event_type_norm").alias("event_type"),
        F.col("created_at_dt").alias("created_at"),
        F.col("severity_norm").alias("severity"),
        F.col("status_norm").alias("status"),
        F.current_timestamp().alias("_processed_at"),
    )
)

df_oc_silver.write.format("delta").mode("overwrite").saveAsTable("silver.fct_ocorrencias")
print(f"silver.fct_ocorrencias: {df_oc_silver.count()} registros")
df_oc_silver.groupBy("event_type", "status").count().orderBy("count", ascending=False).show()

silver.fct_ocorrencias: 270 registros
+------------+------------+-----+
|  event_type|      status|count|
+------------+------------+-----+
|      Atraso|      Aberto|   35|
|      Atraso|Desconhecido|   29|
|      Atraso|     Fechado|   20|
|   Reembolso|      Aberto|   19|
|Desconhecido|      Aberto|   19|
|       Troca|      Aberto|   18|
|Cancelamento|      Aberto|   17|
|  Reclamação|      Aberto|   16|
|Desconhecido|     Fechado|   13|
|   Reembolso|     Fechado|   12|
|       Troca|     Fechado|   12|
|Cancelamento|Desconhecido|   12|
|       Troca|Desconhecido|   11|
|   Reembolso|Desconhecido|   10|
|  Reclamação|Desconhecido|    8|
|Cancelamento|     Fechado|    7|
|  Reclamação|     Fechado|    7|
|Desconhecido|Desconhecido|    5|
+------------+------------+-----+



## Validação Final — Silver Layer

In [0]:
silver_tables = [
    "silver.dim_regioes",
    "silver.dim_canais",
    "silver.dim_produtos",
    "silver.dim_clientes",
    "silver.dim_vendedores",
    "silver.fct_pedidos_cabecalho",
    "silver.fct_pedidos_itens",
    "silver.fct_entregas",
    "silver.fct_ocorrencias",
]

print("=" * 55)
print("RESUMO — SILVER LAYER (sandbox.silver.*)")
print("=" * 55)
for t in silver_tables:
    count = spark.table(t).count()
    print(f"  {t:<45} {count:>6} registros")

RESUMO — SILVER LAYER (sandbox.silver.*)
  silver.dim_regioes                                 6 registros
  silver.dim_canais                                  7 registros
  silver.dim_produtos                               71 registros
  silver.dim_clientes                              183 registros
  silver.dim_vendedores                             40 registros
  silver.fct_pedidos_cabecalho                     403 registros
  silver.fct_pedidos_itens                         995 registros
  silver.fct_entregas                              322 registros
  silver.fct_ocorrencias                           270 registros
